In [ ]:
import funciones as f
from PIL import Image
import numpy as np
import pandas as pd
from scipy.ndimage import median_filter, binary_closing, binary_opening
from skimage.morphology import disk, remove_small_holes
from scipy import ndimage
import matplotlib.pyplot as plt
from skimage.util import img_as_float
from skimage.measure import label, regionprops
from skimage import io, color
from scipy.ndimage import distance_transform_edt
from skimage.morphology import skeletonize

In [ ]:
#--------------------------------------------------------------
# Paso 0: cargar imagen
#--------------------------------------------------------------
img_raw = f.cargar_imagen('img/pensamientos_paso3.png')
# img_raw = funciones.cargar_imagen(r"img/flores de lupino.png")
h, w, c = img_raw.shape

In [ ]:
def separar_areas_por_color(imagen):

    colores_flat = imagen.reshape(-1, 3)

    colores_unicos = np.unique(colores_flat, axis=0)

    mapa_regiones = np.zeros((h, w), dtype=np.int32)

    regiones = []

    region_id_global = 1

    for color in colores_unicos:
        # máscara del color actual
        mascara = np.all(imagen == color, axis=-1)

        # componentes conectados
        etiquetas = label(mascara, connectivity=1)

        props = regionprops(etiquetas)

        for prop in props:

            region_mask = etiquetas == prop.label

            mapa_regiones[region_mask] = region_id_global

            regiones.append({
                "region_id": region_id_global,
                "color_r": int(color[0]),
                "color_g": int(color[1]),
                "color_b": int(color[2]),
                "area": int(prop.area),
                "bbox": prop.bbox
            })

            region_id_global += 1

    df_regiones = pd.DataFrame(regiones)

    return mapa_regiones, df_regiones

def calcular_ancho_promedio_region(mask):
    """
    Calcula el ancho promedio de una región binaria.

    Parámetros
    ----------
    mask : ndarray bool
        Máscara de la región.

    Retorna
    -------
    ancho_promedio : float
    """

    # distancia al borde
    dist = distance_transform_edt(mask)

    # ancho local
    ancho_local = dist * 2

    # solo píxeles internos
    valores = ancho_local[mask]

    return valores.mean()

def calcular_largo_region(mask):
    """
    Calcula el largo aproximado de una región usando skeleton.

    Parámetros
    ----------
    mask : ndarray bool

    Retorna
    -------
    largo : float
    """

    # skeleton de 1 pixel
    skel = skeletonize(mask)

    # coordenadas del skeleton
    yx = np.argwhere(skel)

    if len(yx) == 0:
        return 0

    # cantidad de píxeles del skeleton
    largo = len(yx)

    return float(largo)

In [ ]:

mapa_regiones, df_regiones = separar_areas_por_color(img_raw)

anchos = []
largos = []

for region_id in df_regiones["region_id"]:

    mask = mapa_regiones == region_id

    ancho = calcular_ancho_promedio_region(mask)
    largo = calcular_largo_region(mask)

    anchos.append(ancho)
    largos.append(largo)


df_regiones["ancho_promedio"] = anchos
df_regiones["largo"] = largos

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(mapa_regiones, cmap='nipy_spectral')
plt.title("Mapa de regiones")
plt.axis('off')
plt.show()

In [ ]:
region_id = 3

mask = mapa_regiones == region_id

plt.imshow(mask, cmap='gray')
plt.title(f"Region {region_id}")
plt.axis('off')
plt.show()

In [ ]:
id_regiones_angostas = df_regiones[df_regiones['ancho_promedio'] <= 5]['region_id'].unique()
# TODO 1: crear una funcion que elimine (una a otra mas grande) las reigones con poco ancho y harto largo
# TODO 2: revisar el resultado porque aun hay regiones de 1px

In [ ]:
# for region_id in df_regiones["region_id"]:
for region_id in id_regiones_angostas:

    # máscara de la región
    mask = mapa_regiones == region_id

    # mostrar
    plt.figure(figsize=(6,6))
    plt.imshow(mask, cmap='gray')

    area = df_regiones[df_regiones['region_id'] == region_id]['area'].tolist()[0]
    ancho = df_regiones[df_regiones['region_id'] == region_id]['ancho_promedio'].tolist()[0]
    largo = df_regiones[df_regiones['region_id'] == region_id]['largo'].tolist()[0]
     

    plt.title(f"ID:{region_id}, area:{area}, ancho:{ancho:.2f}, largo:{largo}")
    plt.axis('off')

    plt.show(block=False)

    # esperar tecla
    tecla = input("ENTER = siguiente | q = salir : ")

    plt.close()

    if tecla.lower() == 'q':
        break

In [ ]:
# mapa_colores = dict(zip(df_regiones_limpio['region_id'], df_regiones_limpio['color_id']))
# img_regiones_filtrado = np.vectorize(mapa_colores.get)(mapa_regiones_limpio.reshape(-1))
# img_regiones_filtrado = img_regiones_filtrado.reshape(h, w)
# img_regiones_filtradas = colores_paleta[img_regiones_filtrado - 1]
# Image.fromarray(img_regiones_filtradas).save('img/pensamientos_medio_procesado.png')


In [ ]:
df_regiones.plot(x='region_id', y='ancho_promedio')